# 00 · Meet Mario

Greenstone, from Python: boot Super Mario Bros., hold buttons, watch the bytes move.

`gs.boot("smb1")` gives a console at the first frame of play (the ROM comes from greenstone's `roms2/`).
- `nes.step(buttons, frames=1)` runs frames with buttons held (`gs.RIGHT | gs.A`, …).
- `nes.play(buttons_array, pictures=False)` runs one frame per entry and returns the memory after each, `(frames, 2048)`; add `pictures=True` to get the pictures too.
- `nes.ram`, `nes.peek(addr)`, `nes.poke(addr, value)` read and write memory.
- `nes.picture()` returns the screen; `nes.save()` / `nes.load(state)` keep a moment to come back to.
- `nes.frame` counts the frames since boot.

The bytes the videos use (from greenstone's `src/nes/gym/series/ram.rs`):

| address | what | units |
|---|---|---|
| `0x000E` | player state | `08` = normal play |
| `0x0755` | Mario's position on screen, as the scroll code reads it | pixels from the left edge |
| `0x0057` | horizontal speed | signed, sixteenths of a pixel a frame |
| `0x0775` | how far the screen scrolled this frame | pixels |
| `0x001D` | in the air? | `0` ground, `1` jumping, `2` falling |
| `0x00CE` | height | pixels from the top |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

import greenstone as gs

PLAY, POSITION, SPEED, SCROLL, FLOAT, Y = 0x000E, 0x0755, 0x0057, 0x0775, 0x001D, 0x00CE


def show(picture, title=None):
    plt.figure(figsize=(4, 3.75))
    plt.imshow(picture)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()


def held(*parts):
    """A button track from (buttons, frames) pairs: held(gs.RIGHT, 60, gs.RIGHT | gs.A, 20)."""
    return np.concatenate([np.full(n, b, np.uint8) for b, n in zip(parts[::2], parts[1::2])])

In [ ]:
nes = gs.boot("smb1")
show(nes.picture(), f"frame {nes.frame}")

## Hold Right for three seconds

In [ ]:
start = nes.save()
ram, pictures = nes.play(held(gs.RIGHT, 180), pictures=True)

fig, ax = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
ax[0].plot(ram[:, POSITION]); ax[0].set_ylabel("position (px)")
ax[1].plot(ram[:, SPEED].astype(np.int8) / 16); ax[1].set_ylabel("speed (px/frame)")
ax[2].plot(ram[:, SCROLL]); ax[2].set_ylabel("scrolled (px)")
ax[2].set_xlabel("frame")
plt.show()

Scrub through it: the picture after each frame, beside the bytes.

In [ ]:
def look(f):
    r = ram[f]
    show(pictures[f], f"frame {f}: position {r[POSITION]}, speed {np.int8(r[SPEED]) / 16:.2f}, scrolled {r[SCROLL]}")

interact(look, f=IntSlider(0, 0, len(ram) - 1));

## Play

Some things to find out. `nes.load(start)` puts you back where the run above began.

1. **Running.** Hold `gs.RIGHT | gs.B`. What's Mario's top speed walking, and running? (002 says 1.5 and 2.5 pixels a frame.)
2. **The line.** At what position does the screen start scrolling instead of Mario moving? (002 says 80.)
3. **The lag.** Does the picture show the same moment as the memory? Find the first frame the position byte changes, and the first frame Mario's sprite moves on screen. (The series measured one frame. Hint: compare `pictures[f]` with `pictures[f - 1]`.)
4. **Jumping.** Add `gs.A` partway through. Watch `FLOAT` and `Y`. Does a longer press jump higher?
5. **Poke something.** What happens if you `nes.poke(SPEED, 40)` mid-run?